# 4단계: Gemma-2B full fine-tuning (논문 Octopus-3 조건)
학습 코드는 `src/train.py` 에 있다. 이 노트북은 GitHub 에서 코드를 받아 **실행만** 한다.
(코드를 고칠 땐 랩탑에서 고치고 push → 여기서 `git pull`)

- 런타임: **A100 GPU + 고용량 RAM (GPU 80GB)**. 40GB 에서는 일반 AdamW 가 메모리에 안 들어간다. 🔑 Secrets에 `HF_TOKEN` 등록 (`google/gemma-2b` 라이선스 동의한 계정)
- 결과는 Google Drive 의 `MyDrive/octopus_v2/runs/` 에 저장
- 학습 설정: 논문 3.4절 (AdamW, lr 5e-5, warm-up 10, linear, 3 epochs) + batch 16 (논문에 없는 추가)

## 준비 (그냥 실행)

In [1]:
# 랩탑에서 확인한 버전으로 고정 (최신 5.x 는 API 가 바뀌어 에러가 난다)
# 처음 설치한 뒤에는 '런타임 > 세션 다시 시작' 후 이 칸부터 다시 실행
!pip -q install transformers==4.56.2 accelerate bitsandbytes
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.8 MB/s eta 0:00:00
Mounted at /content/drive
Cloning into '/content/octopus_v2_reproduce'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 120 (delta 54), reused 83 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 1.91 MiB | 3.59 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/octopus_v2_reproduce
Already up to date.
fd99a4b (HEAD -> main, origin/main, origin/HEAD) 4단계 훈련 노트북 생성


In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print("transformers", transformers.__version__)

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
transformers 5.18.0


---
## 4-3. 학습 전 점검
모델을 올리고, 단어 뜻 칸을 늘리고, 학습 전 loss 를 본다.

**예측**: 칸은 몇 → 몇? `<nexa_*>` 자리 loss 는 나머지 정답 자리보다 높을까 낮을까?
> (여기에 예측)

확인할 것
- `[모델] 단어 뜻 칸 256000 → 256022`
- `<nexa_*>` 자리 loss 가 나머지보다 높아야 정상 (아직 뜻이 없는 새 단어)
- `[GPU] 모델만 올렸을 때 ~10 GiB` (2.5B × fp32 4바이트)

In [ ]:
!python -m src.train --check-only

---
## 4-4. 짧게 시험 학습 (30 step)
loss 가 내려가는지, 메모리가 버티는지만 본다. 저장은 `/content` (Drive 아님).

80GB 에서는 옵션 없이 된다 (최대 67.3 GiB). **메모리 부족(CUDA out of memory)이 나면** 위에서부터 하나씩 추가
(2번 8-bit AdamW 는 논문의 AdamW 와 달라지므로 쓰면 README 에 기록)
1. `--batch-size 4 --grad-accum 4` : 한 번에 4개씩 4번 모아서 갱신 → 실제 batch 는 그대로 16
2. `--optim adamw_bnb_8bit` : AdamW 가 기억하는 값을 8bit 로 (약 20GB → 5GB)
3. `--gradient-checkpointing` : 중간 계산을 버렸다 다시 계산 (느려짐)

쓴 옵션은 아래 본 학습에도 똑같이 붙이고, README 에 기록한다.

In [3]:
EXTRA = ""   # 예: "--batch-size 4 --grad-accum 4 --optim adamw_bnb_8bit"
!python -m src.train --max-steps 30 --output-dir /content/runs/try {EXTRA}

config.json: 100% 627/627 [00:00<00:00, 2.48MB/s]
[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.
tokenizer_config.json: 100% 33.6k/33.6k [00:00<00:00, 53.2MB/s]

tokenizer.json: downloading bytes:  35% 6.18M/17.5M [00:01<00:02, 4.06MB/s]
tokenizer.json: downloading bytes: 100% 7.44M/7.44M [00:01<00:00, 4.78MB/s,  712kB/s  ]
tokenizer.json: reconstructing file: 100% 17.5M/17.5M [00:01<00:00, 11.3MB/s, 1.68MB/s  ]
special_tokens_map.json: 100% 636/636 [00:00<00:00, 3.18MB/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/octopus_v2_reproduce/src/train.py", line 153, in <module>
    main()
    ~~~~^^
  File "/content/octopus_v2_reproduce/src/train.py", line 91, in main
    nexa_ids = add_functional_tokens(tokenizer)


---
## 4-5. 본 학습 (3 epoch)
4,000개 ÷ 16 = 250 step × 3 = 750 step. 끝나면 모델·토크나이저·loss 기록이 Drive 에 저장된다.

In [ ]:
RUN_DIR = "/content/drive/MyDrive/octopus_v2/runs/full"
!python -m src.train --output-dir {RUN_DIR} {EXTRA}

In [ ]:
# loss 곡선
import json, matplotlib.pyplot as plt
logs = [l for l in json.load(open(f"{RUN_DIR}/log_history.json")) if "loss" in l]
plt.plot([l["step"] for l in logs], [l["loss"] for l in logs]); plt.xlabel("step"); plt.ylabel("train loss"); plt.yscale("log"); plt.show()

---
## 4-6. 저장한 모델 다시 불러서 몇 개 생성해 보기
정확도 측정은 5단계. 여기서는 모양만 본다. `<nexa_end>` 에서 멈추게 한다.
`skip_special_tokens=False` 로 decode (True 면 `<nexa_*>` 가 지워진다).

In [ ]:
# 세션을 다시 시작했으면 4-5(학습)를 다시 돌리지 말고 이 칸만 실행 (모델은 Drive 에 있다)
RUN_DIR = "/content/drive/MyDrive/octopus_v2/runs/full"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.preprocess.prompt import build_prompt, load_jsonl
from pathlib import Path

tok = AutoTokenizer.from_pretrained(RUN_DIR)
model = AutoModelForCausalLM.from_pretrained(RUN_DIR, dtype=torch.bfloat16, device_map="cuda").eval()
stop = [tok.convert_tokens_to_ids("<nexa_end>"), tok.eos_token_id]

test = load_jsonl(Path("data/test.jsonl"))
for row in test[:3] + test[-3:]:
    ids = tok(build_prompt(row["query"]), return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=64, do_sample=False, eos_token_id=stop)
    pred = tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=False)
    print(f"Q: {row['query']}
   정답: {row['call']}
   예측:{pred}
")